In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file = pd.read_csv("/content/landslide_raw.csv")
print("Shape:",file.shape)
file.head()

Shape: (4789, 11)


,Sl.No.,Slide_No,State,District,Slide_Name,NH_SH_Location,Latitude,Longitude,Material_Involved,Movement_Type,History
0,LANDSLIDE INVENTORY (Field vaidated),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,ASM/HKN/83D07/2020/2,Assam,Hailakandi,Kukinala slide,Kukinala,24.27,92.50,Debris,Slide,NaN
2,2,AS/HKN/83D11/2020/1,Assam,Hailakandi,Jalnachaura slide,Jalnachaura,24.31,92.56,Debris,Slide,NaN
3,3,AS/HKN/83D10/2020/6,Assam,Hailakandi,Nandagram slide,Nandagram Pt. I,24.32,92.51,Debris,Slide,NaN
4,4,AS/HKN/83D11/2020/4,Assam,Hailakandi,Nishkar slide,Mahapur,24.43,92.55,Debris,Slide,NaN


In [ ]:
print("data type of each column:\n",file.dtypes)
print("\n empty cell in each column:\n",file.isnull().sum())
print("\nNo of duplicate row:\n",file.duplicated().sum())

data type of each column:
 Sl.No.                object
Slide_No              object
State                 object
District              object
Slide_Name            object
NH_SH_Location        object
Latitude             float64
Longitude            float64
Material_Involved     object
Movement_Type         object
History               object
dtype: object

 empty cell in each column:
 Sl.No.                  0
Slide_No                1
State                   1
District                1
Slide_Name           2916
NH_SH_Location        272
Latitude                1
Longitude               2
Material_Involved      33
Movement_Type          12
History              4200
dtype: int64

No of duplicate row:
 0


In [ ]:
file["Sl.No."]=pd.to_numeric(file["Sl.No."],errors="coerce").astype("Int64")
print("\ndata type of sno\n",file["Sl.No."].dtype)
print("\nNo of empty cell\n",file['Sl.No.'].isnull().sum())
missing_sno = file[file["Sl.No."].isnull()]
print("\nmissing row\n",missing_sno)


data type of sno
 Int64

No of empty cell
 1

missing row
    Sl.No. Slide_No State District Slide_Name NH_SH_Location  Latitude  \
0    <NA>      NaN   NaN      NaN        NaN            NaN       NaN   

   Longitude Material_Involved Movement_Type History  
0        NaN               NaN           NaN     NaN  


In [ ]:
file = file.dropna(subset=["Sl.No."]).reset_index(drop=True)
print("Remaining nulls in Sl.No.:", file["Sl.No."].isnull().sum())
print("New shape:", file.shape)

Remaining nulls in Sl.No.: 0
New shape: (4788, 11)


In [ ]:
print("\nall unique Material_Involved:\n", file["Material_Involved"].unique())
print("\nall unique Movement_Type:\n", file["Movement_Type"].unique())


all unique Material_Involved:
 ['Debris' 'Rock' 'Earth' 'Soil' 'Rock cum debris' 'Rock cum Debris'
 'Debris cum earth' 'Earth cum Rock' 'Soil cum debris' nan
 'Rock-cum- debris' 'Soil and Rock' 'Debris/unconsolidat\ned material'
 'Unconsolidated rock\ndebris and sandy soil.'
 'Debris and\nunconsolidated\nmaterial'
 'Debris/unconsolidat\ned materials' 'Debris/unconso\nlidated material'
 'Debris\n/unconsolidat\ned material' 'Soil / Earth' 'Soil/Earth'
 'Soil and boulders' 'Earth / soil' 'Earth/Soil/Debris' 'soil'
 'Rock-cum-Debris' 'rock']

all unique Movement_Type:
 ['Slide' 'Flow' 'Fall' 'Subsidence' 'Topple' nan 'Falls'
 'Slide the loose\nsoil from top\n(already cut in\nthe area) due\nto heavy rain.'
 'Creep'
 'Initially started\nas a\nsubsidence,\nnow movement\nof slope mass\nhas already\nbeen initiated.'
 'slide' 'Shallow\nrotational'
 'Toppled the\nwedged\nweathered\nrocks blocks.' 'NIL' 'flow'
 'Debris slide\nand subsidence' 'Translation\n(Debris flow)'
 'Debris slide' 'Rock fall

In [ ]:
import re

def clean_text(x):
    if pd.isna(x):
        return "Unknown"
    x = str(x).replace("\n", " ").strip()
    x = re.sub(r"\s+", " ", x)
    return x

file["Material_Involved"] = file["Material_Involved"].apply(clean_text)
file["Movement_Type"] = file["Movement_Type"].apply(clean_text)
print("\nall unique Material_Involved:\n", file["Material_Involved"].unique())
print("\nall unique Movement_Type:\n", file["Movement_Type"].unique())


all unique Material_Involved:
 ['Debris' 'Rock' 'Earth' 'Soil' 'Rock cum debris' 'Rock cum Debris'
 'Debris cum earth' 'Earth cum Rock' 'Soil cum debris' 'Unknown'
 'Rock-cum- debris' 'Soil and Rock' 'Debris/unconsolidat ed material'
 'Unconsolidated rock debris and sandy soil.'
 'Debris and unconsolidated material' 'Debris/unconsolidat ed materials'
 'Debris/unconso lidated material' 'Debris /unconsolidat ed material'
 'Soil / Earth' 'Soil/Earth' 'Soil and boulders' 'Earth / soil'
 'Earth/Soil/Debris' 'soil' 'Rock-cum-Debris' 'rock']

all unique Movement_Type:
 ['Slide' 'Flow' 'Fall' 'Subsidence' 'Topple' 'Unknown' 'Falls'
 'Slide the loose soil from top (already cut in the area) due to heavy rain.'
 'Creep'
 'Initially started as a subsidence, now movement of slope mass has already been initiated.'
 'slide' 'Shallow rotational' 'Toppled the wedged weathered rocks blocks.'
 'NIL' 'flow' 'Debris slide and subsidence' 'Translation (Debris flow)'
 'Debris slide' 'Rock fall' 'Planar/Wedg

In [ ]:
def map_material(x):
    x = x.lower()
    has_debris = "debris" in x or "unconsolidat" in x
    has_rock = "rock" in x
    has_earth = "earth" in x
    has_soil = "soil" in x
    has_boulder = "boulder" in x

    tags = []
    if has_debris: tags.append("Debris")
    if has_rock: tags.append("Rock")
    if has_earth: tags.append("Earth")
    if has_soil: tags.append("Soil")
    if has_boulder: tags.append("Boulders")

    if not tags:
        return "Unknown"
    return " cum ".join(tags)

file["Material_Involved_Clean"] = file["Material_Involved"].apply(map_material)
print(file["Material_Involved_Clean"].value_counts())

Material_Involved_Clean
Debris                       2045
Rock                         1960
Debris cum Rock               381
Earth                         230
Soil                          110
Unknown                        32
Debris cum Earth                9
Rock cum Earth                  5
Debris cum Soil                 5
Debris cum Rock cum Soil        4
Earth cum Soil                  4
Rock cum Soil                   1
Soil cum Boulders               1
Debris cum Earth cum Soil       1
Name: count, dtype: int64


In [ ]:
def map_movement(x):
    x = x.lower()
    if "topple" in x: return "Topple"
    if "subsid" in x or "sink" in x: return "Subsidence"
    if "creep" in x: return "Creep"
    if "fall" in x: return "Fall"
    if "flow" in x: return "Flow"
    if "rotational" in x: return "Shallow Rotational"
    if "wedge" in x or "planar" in x: return "Planar/Wedge Failure"
    if "complex" in x: return "Complex"
    if "slide" in x or "sldies" in x or "slides" in x: return "Slide"
    if x in ["nil", "unknown", ""]: return "Unknown"
    return "Other"

file["Movement_Type_Clean"] = file["Movement_Type"].apply(map_movement)
print(file["Movement_Type_Clean"].value_counts())

Movement_Type_Clean
Slide                   4454
Fall                     181
Flow                      58
Subsidence                37
Unknown                   22
Topple                    17
Planar/Wedge Failure      13
Creep                      4
Shallow Rotational         1
Complex                    1
Name: count, dtype: int64


In [ ]:
print("Slide_No duplicates:", file["Slide_No"].duplicated().sum())
print("\nState unique:", file["State"].unique())
print("\nDistrict unique:", file["District"].unique())

print("\nSlide_Name nulls:", file["Slide_Name"].isnull().sum())
print("NH_SH_Location nulls:", file["NH_SH_Location"].isnull().sum())

out_of_range = file[(file["Latitude"] < 22) | (file["Latitude"] > 29) |
                     (file["Longitude"] < 88) | (file["Longitude"] > 97)]
print("\nSuspicious coordinates:", len(out_of_range))

Slide_No duplicates: 23

State unique: ['Assam' 'Arunachal Pradesh' '-Arunachal Pradesh' 'Andhra Pradesh'
 'Himachal Pradesh']

District unique: ['Hailakandi' 'Karimganj' 'Cachar' 'Dima Hasao' 'West Karbi Anglong'
 'Kamrup (Rural)' 'Karbi Anglong' 'Nagaon' 'Goalpara' 'Kamrup'
 'Kamrup (Metro)' 'Morigaon' 'Bongaigaon' 'Jorhat' 'Udalguri' 'Sonitpur'
 'Dima Hasao (N.C. Hills)' 'West Khasi Hills' 'Longding' 'Tirap'
 'West Kameng' 'Papum Pare' 'Changlang' 'East Kameng' 'Lower Subansiri'
 'Pakke Kessang' 'Upper Siang' 'Kra Daadi' 'East Siang' 'Tawang'
 'West Siang' 'Kamle' 'Kurung Kumey' 'Lower Siang' 'Upper Subansiri'
 'Lohit' 'Lepa rada' 'Lower dibang valley' 'Shi-Yomi' 'Anjaw' 'Siang'
 'Lower Dibang Valley' 'Dibang Valley' 'Visakhapatanam' 'Sirmur' 'Solan'
 'Shimla' 'Mandi' 'Kullu' 'Bilaspur']

Slide_Name nulls: 2915
NH_SH_Location nulls: 271

Suspicious coordinates: 2708


In [ ]:
print(file["State"].value_counts())

State
Himachal Pradesh      2679
Arunachal Pradesh     1220
Assam                  857
Andhra Pradesh          29
-Arunachal Pradesh       3
Name: count, dtype: int64


In [ ]:
ner_states = ["Assam", "Arunachal Pradesh", "Meghalaya", "Sikkim",
              "Nagaland", "Manipur", "Mizoram", "Tripura"]

file["State"] = file["State"].astype(str).str.strip().str.lstrip("-")
file = file[file["State"].isin(ner_states)].reset_index(drop=True)

print("Shape after NER filter:", file.shape)
print(file["State"].value_counts())

Shape after NER filter: (2080, 13)
State
Arunachal Pradesh    1223
Assam                 857
Name: count, dtype: int64


In [ ]:
print("no of empty latitude:", file["Latitude"].isnull().sum())
print("no of empty longitidue:", file["Longitude"].isnull().sum())

file = file.dropna(subset=["Latitude","Longitude"]).reset_index(drop=True)
print("Shape after dropping missing coords:", file.shape)

no of empty latitude: 0
no of empty longitidue: 1
Shape after dropping missing coords: (2079, 13)


In [ ]:
dupes = file[file["Slide_No"].duplicated(keep=False)]
print("Duplicate Slide_No rows:", len(dupes))
dupes.sort_values("Slide_No")

Duplicate Slide_No rows: 40


,Sl.No.,Slide_No,State,District,Slide_Name,NH_SH_Location,Latitude,Longitude,Material_Involved,Movement_Type,History,Material_Involved_Clean,Movement_Type_Clean
1215,1217,AR/83E/12/2018/06,Arunachal Pradesh,Papum Pare,Sanki-II,MowII-Senkicircular road,27.102150,93.613710,Debris/unconsolidat ed material,Debris slide,NaN,Debris,Slide
916,918,AR/83E/12/2018/06,Arunachal Pradesh,Papum Pare,NaN,NaN,27.102150,93.613710,Debris/unconso lidated material,Debris slide,NaN,Debris,Slide
901,903,AR/83E/12/2021/03,Arunachal Pradesh,Papum Pare,NaN,NaN,27.083117,93.601735,Debris and unconsolidated material,Debris slide,NaN,Debris,Slide
1212,1214,AR/83E/12/2021/03,Arunachal Pradesh,Papum Pare,Chandranagar,"NH415, Chandranagar, Itanagar",27.083117,93.601735,Debris and unconsolidated material,Debris slide,NaN,Debris,Slide
1214,1216,AR/83E/12/2021/05,Arunachal Pradesh,Papum Pare,Sanki-I,Mow2-Senkiview road section,27.103750,93.615290,Debris,Debris slide,NaN,Debris,Slide
924,926,AR/83E/12/2021/05,Arunachal Pradesh,Papum Pare,NaN,NaN,27.103750,93.615290,Debris /unconsolidat ed material,Debris slide,"2020,2021",Debris,Slide
1211,1213,AR/83E/12/2021/2,Arunachal Pradesh,Papum Pare,D-Sector,"NH415, Itanagar city",27.095890,93.622840,Debris/unconsolidat ed materials,Debris slide,NaN,Debris,Slide
911,913,AR/83E/12/2021/2,Arunachal Pradesh,Papum Pare,NaN,NaN,27.095890,93.622840,Debris/unconsolidat ed materials,Debris slide,NaN,Debris,Slide
1441,1443,AR/ANJ/91D08/2014/15,Arunachal Pradesh,Anjaw,78 km slide,Tezu-Hayuliang road,28.059390,96.483970,Debris,Slide,2014,Debris,Slide
1426,1428,AR/ANJ/91D08/2014/15,Arunachal Pradesh,Anjaw,78 km slide,Tezu-Hayuliang road,28.016610,96.441720,Rock cum Debris,Complex,2013,Debris cum Rock,Complex


In [ ]:
cols_to_check = ["Slide_Name", "NH_SH_Location", "History"]
file["completeness"] = file[cols_to_check].notna().sum(axis=1)
file = file.sort_values("completeness", ascending=False)
file = file.drop_duplicates(subset=["Slide_No"], keep="first").reset_index(drop=True)
file = file.drop(columns=["completeness"])
print("Shape after smart dedup:", file.shape)
print("Remaining duplicate Slide_No:", file["Slide_No"].duplicated().sum())

Shape after smart dedup: (2059, 13)
Remaining duplicate Slide_No: 0


In [ ]:
print("no of missing latitude:", file["Latitude"].isnull().sum())
print("no of missing langitude:", file["Longitude"].isnull().sum())
file = file.dropna(subset=["Latitude","Longitude"]).reset_index(drop=True)
print("Shape after deleting null:", file.shape)

no of missing latitude: 0
no of missing langitude: 0
Shape after deleting null: (2059, 13)


In [ ]:
print("no of row with Null Slide_Name:", file["Slide_Name"].isna().sum())
print("no of row with Null NH_SH_Location:", file["NH_SH_Location"].isna().sum())

no of row with Null Slide_Name: 217
no of row with Null NH_SH_Location: 178


In [ ]:
file["Slide_Name"] = file["Slide_Name"].fillna("Unknown")
file["NH_SH_Location"] = file["NH_SH_Location"].fillna("Unknown")
print("Remaining nulls:\n", file[["Slide_Name","NH_SH_Location"]].isnull().sum())

Remaining nulls:
 Slide_Name        0
NH_SH_Location    0
dtype: int64


In [ ]:
print("no of not null history:", file["History"].notna().sum())
print(file["History"].dropna().head(10))
file["has_history_date"] = file["History"].notna().astype(int)

no of not null history: 552
0       18 May 2016
1       18 May 2016
2    02nd June 2020
3      02 June 2020
4      03 June 2010
5      05 June 2020
6     02 April 2010
7              2013
8              2013
9              2013
Name: History, dtype: object


In [ ]:
file["event_year"] = file["History"].str.extract(r"(\d{4})")
file["event_year"] = pd.to_numeric(file["event_year"], errors="coerce")

print(file["event_year"].describe())
print(file["event_year"].value_counts().sort_index())

count     551.000000
mean     2016.286751
std         5.953867
min      1950.000000
25%      2014.000000
50%      2016.000000
75%      2019.500000
max      2026.000000
Name: event_year, dtype: float64
event_year
1950.0      2
1985.0      1
1998.0      2
2005.0      3
2006.0      1
2008.0     15
2009.0      1
2010.0      9
2011.0      5
2012.0     15
2013.0     47
2014.0     81
2015.0     18
2016.0    159
2017.0     20
2018.0     14
2019.0     20
2020.0     34
2021.0      3
2022.0     51
2023.0      2
2024.0     27
2025.0     18
2026.0      3
Name: count, dtype: int64


In [ ]:
print(file[file["event_year"] == 2026][["Slide_No","State","District","History"]])

    Slide_No              State        District                    History
399     7583  Arunachal Pradesh     Lower Siang               28 June 2026
400     7386  Arunachal Pradesh      Papum Pare              16 March 2026
845     7460              Assam  Kamrup (Metro)  31 May 2026, 15 July 2025


In [ ]:
file = file.drop(columns=["History"])
print("Shape after dropping raw History:", file.shape)

Shape after dropping raw History: (2059, 14)


In [ ]:
print("no of null District :", file["District"].isnull().sum())
print("unique district",file["District"].unique())
print("no of Unique districts:", file["District"].nunique())

no of null District : 0
unique district ['Hailakandi' 'Karimganj' 'Cachar' 'Anjaw' 'Lohit' 'Dima Hasao'
 'Kra Daadi' 'Tawang' 'Papum Pare' 'East Siang' 'Lower Subansiri'
 'West Kameng' 'Kamrup' 'Kamrup (Metro)' 'West Karbi Anglong'
 'Lower Dibang Valley' 'Dima Hasao (N.C. Hills)' 'Kamrup (Rural)' 'Nagaon'
 'Lower Siang' 'West Siang' 'Upper Siang' 'Shi-Yomi' 'Karbi Anglong'
 'Goalpara' 'Morigaon' 'Jorhat' 'Bongaigaon' 'Sonitpur' 'Dibang Valley'
 'Tirap' 'West Khasi Hills' 'Longding' 'Changlang' 'East Kameng'
 'Pakke Kessang' 'Kamle' 'Kurung Kumey' 'Upper Subansiri' 'Lepa rada'
 'Lower dibang valley' 'Siang' 'Udalguri']
no of Unique districts: 43


In [ ]:
print("Final shape:", file.shape)
print("\nNulls per column:\n", file.isnull().sum())
print("\nDtypes:\n", file.dtypes)
print("\nState counts:\n", file["State"].value_counts())

Final shape: (2059, 14)

Nulls per column:
 Sl.No.                        0
Slide_No                      0
State                         0
District                      0
Slide_Name                    0
NH_SH_Location                0
Latitude                      0
Longitude                     0
Material_Involved             0
Movement_Type                 0
Material_Involved_Clean       0
Movement_Type_Clean           0
has_history_date              0
event_year                 1508
dtype: int64

Dtypes:
 Sl.No.                       Int64
Slide_No                    object
State                       object
District                    object
Slide_Name                  object
NH_SH_Location              object
Latitude                   float64
Longitude                  float64
Material_Involved           object
Movement_Type               object
Material_Involved_Clean     object
Movement_Type_Clean         object
has_history_date             int64
event_year                 flo

In [ ]:
file.to_csv("landslide_cleaned_ner.csv", index=False)
print("Saved:", file.shape)

Saved: (2059, 14)


In [ ]:
file["District"] = file["District"].astype(str).str.strip().str.title()
district_fix_map = {
    "Dima Hasao (N.C. Hills)": "Dima Hasao",
    "Lower Dibang Valley": "Lower Dibang Valley",
    "Lepa Rada": "Lepa Rada",
}

file["District"] = file["District"].replace(district_fix_map)

print("No of unique districts after fix:", file["District"].nunique())
print(sorted(file["District"].unique()))

No of unique districts after fix: 41
['Anjaw', 'Bongaigaon', 'Cachar', 'Changlang', 'Dibang Valley', 'Dima Hasao', 'East Kameng', 'East Siang', 'Goalpara', 'Hailakandi', 'Jorhat', 'Kamle', 'Kamrup', 'Kamrup (Metro)', 'Kamrup (Rural)', 'Karbi Anglong', 'Karimganj', 'Kra Daadi', 'Kurung Kumey', 'Lepa Rada', 'Lohit', 'Longding', 'Lower Dibang Valley', 'Lower Siang', 'Lower Subansiri', 'Morigaon', 'Nagaon', 'Pakke Kessang', 'Papum Pare', 'Shi-Yomi', 'Siang', 'Sonitpur', 'Tawang', 'Tirap', 'Udalguri', 'Upper Siang', 'Upper Subansiri', 'West Kameng', 'West Karbi Anglong', 'West Khasi Hills', 'West Siang']


In [ ]:
file.to_csv("landslide_cleaned_ner.csv", index=False)
print("Saved:", file.shape)

Saved: (2059, 14)


In [29]:
import requests
import math
import time

ELEVATION_BASE_URL = "https://api.open-meteo.com/v1/elevation"

def get_elevation(lat: float, lon: float, retries: int = 3, timeout: int = 30):
    params = {"latitude": lat, "longitude": lon}
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(ELEVATION_BASE_URL, params=params, timeout=timeout)
            response.raise_for_status()
            data = response.json()
            return data["elevation"][0] if data.get("elevation") else None
        except requests.exceptions.RequestException as e:
            last_error = e
    return None

def estimate_slope_degrees(lat: float, lon: float, offset: float = 0.01):
    elev_center = get_elevation(lat, lon)
    elev_offset = get_elevation(lat + offset, lon)
    if elev_center is None or elev_offset is None:
        return None
    horizontal_distance_m = offset * 111320
    vertical_diff_m = abs(elev_offset - elev_center)
    if horizontal_distance_m == 0:
        return 0.0
    slope_rad = math.atan(vertical_diff_m / horizontal_distance_m)
    return round(math.degrees(slope_rad), 2)

In [30]:
from tqdm import tqdm

slopes = []
for idx, row in tqdm(file.iterrows(), total=len(file)):
    slope = estimate_slope_degrees(row["Latitude"], row["Longitude"])
    slopes.append(slope)
    time.sleep(0.3)  # thoda rukna zaroori hai, warna API block kar sakti hai

file["slope_deg"] = slopes

100%|██████████| 2059/2059 [32:30<00:00,  1.06it/s]


In [31]:
print("Total rows:", len(file))
print("Missing slope values:", file["slope_deg"].isnull().sum())

Total rows: 2059
Missing slope values: 66


In [32]:
missing_slope = file[file["slope_deg"].isnull()]
print(missing_slope[["Latitude", "Longitude", "State", "District"]])

       Latitude  Longitude              State     District
1956  27.177138  92.576909  Arunachal Pradesh  West Kameng
1957  27.178567  92.583154  Arunachal Pradesh  West Kameng
1958  27.296928  93.088111  Arunachal Pradesh  East Kameng
1959  27.296928  93.088111  Arunachal Pradesh  East Kameng
1960  27.296928  93.088111  Arunachal Pradesh  East Kameng
...         ...        ...                ...          ...
2039  27.083860  92.587656  Arunachal Pradesh  West Kameng
2041  27.059755  93.522666  Arunachal Pradesh   Papum Pare
2042  27.285858  93.798472  Arunachal Pradesh   Papum Pare
2043  27.092699  92.583140  Arunachal Pradesh  West Kameng
2044  27.311571  93.046703  Arunachal Pradesh  East Kameng

[66 rows x 4 columns]


In [33]:
file["slope_deg"] = file.groupby("District")["slope_deg"].transform(
    lambda x: x.fillna(x.median())
)

overall_median = file["slope_deg"].median()
file["slope_deg"] = file["slope_deg"].fillna(overall_median)

print("Final missing:", file["slope_deg"].isnull().sum())

Final missing: 0


In [34]:
file.to_csv("landslide_with_slope.csv", index=False)
print("Saved:", file.shape)

Saved: (2059, 15)
